# 05 · Cambio interanual de vegetación

**Objetivo:** Identificar pérdida y recuperación de cobertura vegetal.

**Datos:** Compuestos Sentinel-2 de dos periodos.

**Relevancia para política ambiental y social:** Permite focalizar alertas de deforestación, restauración o cambio productivo.

**Limitaciones:** El cambio fenológico estacional puede confundirse con cambio de cobertura.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
old = s2_composite("2019-01-01","2019-12-31")
new = s2_composite("2025-01-01","2025-12-31")
old_ndvi = old.normalizedDifference(["B8","B4"])
new_ndvi = new.normalizedDifference(["B8","B4"])
change = new_ndvi.subtract(old_ndvi).rename("NDVI_change")

Map.addLayer(change, {"min":-0.5,"max":0.5,"palette":["red","white","green"]}, "Cambio NDVI")
Map
